In [ ]:
import xarray as xr
import pint_xarray
import matplotlib.pylab as plt
import geopandas as gpd

from pism_terra.grids import sample_points


In [ ]:
carra2 = xr.open_dataset("s3://pism-cloud-data/glacier/input/s4f/climate/zarra2_daily_RGI2000-v7.0-C-01-04374.zarr", 
                     engine="zarr",
                     storage_options={"anon": True})
monthly = carra2["air_temp"].resample(time="1MS").mean()
jja = monthly.sel(time=monthly["time"].dt.month.isin([6, 7, 8]))
jja = xr.merge([jja, carra2.spatial_ref])

carra2_surface = xr.open_dataset("/home/andy/storstrommen/terra/glacier_s4f_input/RGI2000-v7.0-C-01-04374/input/carra2_RGI2000-v7.0-C-01-04374.nc")
glacier = xr.open_dataset("/home/andy/storstrommen/terra/glacier_s4f_input/RGI2000-v7.0-C-01-04374/input/bootfile_RGI2000-v7.0-C-01-04374.nc")
sites_df = gpd.read_file("/home/andy/pism-terra/pism_terra/data/kennicott-points.gpkg")
outlines = gpd.read_file("/home/andy/storstrommen/terra/glacier_s4f_input/RGI2000-v7.0-C-01-04374/input/rgi_RGI2000-v7.0-C-01-04374-G.gpkg")

carra2_surface_monthly = carra2_surface["air_temp"].sel(time="2005")
surface_jja = carra2_surface_monthly.sel(time=monthly["time"].dt.month.isin([6, 7, 8]))
surface_jja = xr.merge([surface_jja, carra2_surface.spatial_ref])

In [ ]:
import numpy as np

R_D, G = 287.05, 9.80665
p = jja["pressure_level"]                       # 1000, 900, 800, 750, 500 hPa, bottom up
T = jja["air_temp"]                            # K

# Anchor: 1000 hPa height from the ISA, ~110 m at sea level pressure 1013.25 hPa.
z0 = 44330.8 * (1.0 - (1000.0 / 1013.25) ** 0.190263)

# Layer thicknesses from the mean temperature of adjacent levels.
T_mean = 0.5 * (T.isel(pressure_level=slice(None, -1)).values + T.isel(pressure_level=slice(1, None)).values)
dz = (R_D / G) * T_mean * np.log(p.values[:-1] / p.values[1:])[None, :, None, None]

z = T.copy(data=np.concatenate([np.full_like(T.values[:, :1], z0), z0 + np.cumsum(dz, axis=1)], axis=1))
z.attrs = {"long_name": "height of the pressure level", "units": "m"}
z.name = "z"

z = xr.merge([z, carra2.spatial_ref])

In [ ]:
carra2_sites = sample_points(jja, sites_df, variables=jja.data_vars).drop_vars(["x", "y"])
carra2_surface_sites = sample_points(surface_jja, sites_df, variables=surface_jja.data_vars).drop_vars(["x", "y"])
surface_sites = sample_points(glacier, sites_df, variables=["surface"])
z_sites = sample_points(z , sites_df) + surface_sites.surface - z0
carra2_surface_sites = carra2_surface_sites.assign_coords(z=surface_sites.surface)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from matplotlib.dates import DateFormatter, date2num


_t = carra2_sites["air_temp"]              # (time, pressure_level, site), K
_z = z_sites["z"]

sites = carra2_sites["site"].values
times = pd.DatetimeIndex(carra2_sites["time"].values)
colors = date2num(times)         # one number per day for the colour scale

surf = carra2_surface_sites
surf_colors = date2num(pd.DatetimeIndex(surf["time"].values))



fig, axes = plt.subplots(1, len(sites), figsize=(4 * len(sites), 6), sharey=True, constrained_layout=True)
for ax, site in zip(np.atleast_1d(axes), sites):
    tz = _t.sel(site=site).transpose("time", "pressure_level").values - 273.15
    zz = _z.sel(site=site).transpose("time", "pressure_level").values
    segments = np.stack([tz, zz], axis=-1)          # (time, level, 2): one polyline per day
    lines = LineCollection(segments, cmap="viridis", linewidths=1, alpha=0.6)
    lines.set_array(colors)
    s = surf.sel(site=site)
    ax.scatter(
        s["air_temp"].values - 273.15,
        np.full(s.sizes["time"], float(s["z"])),
        c=surf_colors,
        cmap="viridis",
        vmin=colors.min(),
        vmax=colors.max(),
        s=60,
        edgecolors="k",
        linewidths=0.8,
        zorder=5,
    )
    ax.scatter(
        tz.ravel(),
        zz.ravel(),
        c=np.repeat(colors, tz.shape[1]),   # one colour per day, repeated per level
        cmap="viridis",
        vmin=colors.min(),
        vmax=colors.max(),
        s=60,
        alpha=0.8,
        linewidths=0,
        zorder=3,
    )
    ax.axhline(float(s["z"]), color="k", ls="--", lw=0.8)
    ax.add_collection(lines)
    #ax.set_xlim(np.nanmin(tz), np.nanmax(tz))
    #ax.set_ylim(np.nanmin(zz), np.nanmax(zz))
    ax.set_title(str(site))
    ax.set_xlabel("Air temperature (°C)")
    ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("Height above sea level (m)")

cbar = fig.colorbar(lines, ax=axes, label="Date", pad=0.02)
cbar.ax.yaxis.set_major_formatter(DateFormatter("%b"))
plt.show()
fig.savefig("carra2_3d.png", dpi=300)

In [ ]:
fig, ax = plt.subplots(1, 1)
outlines.boundary.plot(ax=ax, color="k", lw=0.5)
sites_df.plot("site", ax=ax, legend=True)
fig.savefig("sites.png", dpi=300)

In [ ]:
sites.plot?

In [ ]:
carra2.pressure_level

In [ ]:
ls